# EcoHome Energy Advisor - Agent Run & Evaluation

In this notebook, you'll run the Energy Advisor agent with various real-world scenarios and see how it helps customers optimize their energy usage.

## Learning Objectives
- Create the agent's instructions
- Run the Energy Advisor with different types of questions
- Evaluate response quality and accuracy
- Measure tool usage effectiveness
- Identify areas for improvement
- Implement evaluation metrics

## Evaluation Criteria
- **Accuracy**: Correct information and calculations
- **Relevance**: Responses address the user's question
- **Completeness**: Comprehensive answers with actionable advice
- **Tool Usage**: Appropriate use of available tools
- **Reasoning**: Clear explanation of recommendations


## 1. Import and Initialize

In [1]:
from datetime import datetime
from agent import Agent

In [3]:
# For LLM-as-judge evaluation
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv

load_dotenv()

# Initialize judge LLM for evaluation
def get_judge_llm():
    """Initialize LLM for evaluating responses (LLM-as-judge)."""
    api_key = os.getenv("VOCAREUM_API_KEY")
    return ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0.0,
        base_url="https://openai.vocareum.com/v1",
        api_key=api_key
    )

judge_llm = get_judge_llm()

In [4]:
## TODO: Create the agent's instructions

ECOHOME_SYSTEM_PROMPT = """
You are EcoHome, an AI-powered energy optimization assistant for residential homes.

## Persona
You are a knowledgeable, practical, and trustworthy energy advisor. You help homeowners
understand their energy consumption, reduce costs, and make smarter decisions about
appliances, solar, and utility usage. You speak in a friendly but professional tone, and
always prioritize actionable advice over generic tips.

## Core Capabilities
- Analyze energy usage patterns from historical data
- Query real-time and forecasted electricity prices (time-of-use rates)
- Retrieve weather forecasts and solar irradiance data
- Search energy-saving best practices from a knowledge base
- Calculate potential savings from efficiency improvements
- Provide personalized recommendations ranked by impact and ease

## Available Tools
1. **query_energy_usage** — Get energy consumption data by date range and device type
2. **query_solar_generation** — Get solar production data by date range
3. **get_recent_energy_summary** — Quick 24h (or custom) usage + generation snapshot
4. **get_weather_forecast** — Weather forecast including solar irradiance estimates
5. **get_electricity_prices** — Time-of-use pricing for any date (peak/off-peak rates)
6. **search_energy_tips** — Semantic search through energy-saving best practices
7. **calculate_energy_savings** — Compute kWh/USD savings for optimization scenarios

## Example Questions
Here are example questions that users might ask, showing the types of queries you should recognize and handle:
- "When should I charge my electric car tomorrow to minimize cost and maximize solar power?"
- "What temperature should I set my thermostat for summer to balance comfort and savings?"
- "Suggest three ways I can reduce energy use without sacrificing comfort"
- "How much can I save by running my dishwasher during off-peak hours?"
- "What's the difference between peak and off-peak electricity rates?"
- "My AC runs constantly during hot afternoons. What can I do to reduce energy use?"
- "I have a swimming pool with a 1.5 HP pump. What's the most energy-efficient schedule?"

## Constraints
- Always cite data sources when making claims (e.g., "Based on your usage last
month...")
- Never fabricate numbers — use actual tool outputs or clearly state estimates
- If critical data is missing (e.g., device wattage, tariff info), ask the user before
guessing
- Do not recommend unsafe actions (e.g., modifying electrical systems without a
professional)
- Respect user privacy — never store or share personal data beyond the current session
- If asked about topics outside energy (medical, legal, financial), politely redirect

## Response Best Practices
- Start with the most impactful recommendation, then detail the reasoning
- Include estimated savings (kWh and USD) when data supports it
- Explain tradeoffs (e.g., "This reducespeak usage but may require changing your
schedule")
- Use structured formatting: bullet points for tips, tables for comparisons, bold for
key numbers
- End with a brief summary of what the user should do next
- Always ask if the user wants to explore another area (solar, specific appliances, bill
breakdown)

## User Interaction Style
- Acknowledge the user's goals upfront before diving into analysis
- Use their home data whenever possible — personalization builds trust
- Propose concrete next steps, not just "let me know if you need help"
- If the user asks a vague question, clarify before answering (e.g., "Are you asking
about summer cooling costs or year-round usage?")

## Tone and Voice
- Helpful, encouraging, not judgmental
- Avoid jargon — explain terms like "demand charge" or "peak hours" when first used
- Be concise — respect the user's time

"""

In [5]:
ecohome_agent = Agent(
    instructions=ECOHOME_SYSTEM_PROMPT,
)

In [6]:
response = ecohome_agent.invoke(
    question="When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
    context="Location: San Francisco, CA"
)

In [7]:
print(response["messages"][-1].content)

To minimize costs and maximize solar power when charging your electric car tomorrow (October 7, 2023), here’s what you need to know:

### Electricity Pricing
- **Off-Peak Rates**: $0.08 per kWh (midnight to 6 AM, and 10 PM to midnight)
- **Part-Peak Rates**: $0.12 per kWh (6 AM to 10 AM, and 2 PM to 4 PM)
- **Peak Rates**: $0.18 per kWh (10 AM to 2 PM, and 4 PM to 8 PM)

### Solar Power Generation Forecast
- **Morning (6 AM - 10 AM)**: 
  - 6 AM: 258.1 W/m²
  - 7 AM: 269.4 W/m²
  - 8 AM: 781.7 W/m² (high solar generation)
  - 9 AM: 394.9 W/m²
  - 10 AM: 579.9 W/m²

- **Afternoon (10 AM - 4 PM)**: 
  - 10 AM: 579.9 W/m²
  - 11 AM: 359.1 W/m²
  - 12 PM: 492.7 W/m²
  - 1 PM: 176.4 W/m²
  - 2 PM: 311.7 W/m²
  - 3 PM: 377.6 W/m²

### Recommendations
1. **Charge During Off-Peak Hours**:
   - **Best Time**: Start charging **after 10 PM** or **before 6 AM** to take advantage of the lowest rates ($0.08/kWh).
   
2. **Maximize Solar Power**:
   - If you prefer to charge during the day, aim to st

In [8]:
print("TOOLS:")
for msg in response["messages"]:
    obj = msg.model_dump()
    if obj.get("tool_call_id"):
        print("-", msg.name)

TOOLS:
- get_electricity_prices
- get_weather_forecast


## 2. Define Test Cases

In [9]:
# TODO: Define comprehensive test cases for the Energy Advisor
# Create 10 test cases covering different scenarios:
# - EV charging optimization
# - Thermostat settings
# - Appliance scheduling
# - Solar power maximization
# - Cost savings calculations

In [10]:
test1 = {
        "id": "ev_charging_1",
        "question": "When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
        "expected_response": "The response should contain time recommendation, cost analysis and solar consideration",
        }

In [11]:
test2 = {
        "id": "thermostat_1",
        "question": "What's the best thermostat setting for summer to balance comfort and energy savings?",
        "expected_tools": ["search_energy_tips", "get_weather_forecast"],
        "expected_response": "Should include specific temperature recommendation, estimated savings, and reasoning",
    }

In [12]:
test3 = {
        "id": "appliance_scheduling_1",
        "question": "When is the best time to run my dishwasher and washing machine to save money?",
        "expected_tools": ["get_electricity_prices", "search_energy_tips"],
        "expected_response": "Should recommend off-peak hours and explain time-of-use pricing",
    }

In [13]:
test4 = {
        "id": "solar_optimization_1",
        "question": "How can I maximize my solar power usage during the day?",
        "expected_tools": ["get_weather_forecast", "query_solar_generation", "search_energy_tips"],
        "expected_response": "Should include timing recommendations for high-consumption appliances and solar production patterns",
    }

In [14]:
test5 = {
        "id": "cost_analysis_1",
        "question": "I used 500 kWh last month and my bill was $65. Is that reasonable for a 2-bedroom apartment?",
        "expected_tools": ["query_energy_usage", "get_electricity_prices"],
        "expected_response": "Should compare against average usage, explain rate tiers, and provide context",
    }

In [15]:
test6 = {
        "id": "hvac_1",
        "question": "My AC runs constantly during hot afternoons. What can I do to reduce energy use without sacrificing comfort?",
        "expected_tools": ["get_recent_energy_summary", "get_weather_forecast", "search_energy_tips"],
        "expected_response": "Should identify HVAC as high consumer and provide specific optimization tips",
    }

In [16]:
test7 = {
        "id": "peak_vs_offpeak_1",
        "question": "What's the difference between peak and off-peak electricity rates, and why does it matter for my bill?",
        "expected_tools": ["get_electricity_prices", "search_energy_tips"],
        "expected_response": "Should clearly explain time-of-use tiers, hours, and provide cost impact examples",
    }

In [17]:
test8 = {
        "id": "ev_charging_2",
        "question": "I have a Tesla Model 3 with 60 kWh battery. How much would it cost to fully charge at home?",
        "expected_tools": ["calculate_energy_savings", "get_electricity_prices"],
        "expected_response": "Should calculate cost based on current rates and explain variables (battery size, efficiency)",
    }

In [18]:
test9 = {
        "id": "pool_pump_1",
        "question": "I have a swimming pool with a 1.5 HP pump. What's the most energy-efficient schedule for running it?",
        "expected_tools": ["search_energy_tips", "get_electricity_prices", "calculate_energy_savings"],
        "expected_response": "Should recommend off-peak scheduling, estimate daily/weekly cost, and discuss runtime optimization",
    }

In [19]:
test10 = {
        "id": "energy_audit_1",
        "question": "What are the biggest energy wasters in a typical home and what are the quick fixes I can do myself?",
        "expected_tools": ["search_energy_tips", "get_recent_energy_summary"],
        "expected_response": "Should list common culprits (phantom load, HVAC, lighting), provide actionable tips with estimated savings",
    }

In [20]:
test_cases = [ test1, test2, test3, test4, test5, test6, test7, test8, test9, test10 ]

if len(test_cases) < 10:
    raise ValueError("You MUST have at least 10 test cases")

## 3. Run Agent Tests

In [21]:
CONTEXT = "Location: San Francisco, CA"

In [22]:
# Run the agent tests
# For each test case, call the agent and collect the response
# Store results for evaluation

print("=== Running Agent Tests ===")
test_results = []

for i, test_case in enumerate(test_cases):
    print(f"\nTest {i+1}: {test_case['id']}")
    print(f"Question: {test_case['question']}")
    print("-" * 50)
    
    try:
        # Call the agent
        response = ecohome_agent.invoke(
            question=test_case['question'],
            context=CONTEXT
        )
        
        # Store the result
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': response,
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat()
        }
        test_results.append(result)
                
    except Exception as e:
        print(f"Error: {e}")
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': f"Error: {str(e)}",
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat(),
            'error': str(e)
        }
        test_results.append(result)

print(f"\nCompleted {len(test_results)} tests")


=== Running Agent Tests ===

Test 1: ev_charging_1
Question: When should I charge my electric car tomorrow to minimize cost and maximize solar power?
--------------------------------------------------

Test 2: thermostat_1
Question: What's the best thermostat setting for summer to balance comfort and energy savings?
--------------------------------------------------

Test 3: appliance_scheduling_1
Question: When is the best time to run my dishwasher and washing machine to save money?
--------------------------------------------------

Test 4: solar_optimization_1
Question: How can I maximize my solar power usage during the day?
--------------------------------------------------

Test 5: cost_analysis_1
Question: I used 500 kWh last month and my bill was $65. Is that reasonable for a 2-bedroom apartment?
--------------------------------------------------

Test 6: hvac_1
Question: My AC runs constantly during hot afternoons. What can I do to reduce energy use without sacrificing comfort?

In [23]:
test_results

[{'test_id': 'ev_charging_1',
  'question': 'When should I charge my electric car tomorrow to minimize cost and maximize solar power?',
  'response': {'messages': [SystemMessage(content='\nYou are EcoHome, an AI-powered energy optimization assistant for residential homes.\n\n## Persona\nYou are a knowledgeable, practical, and trustworthy energy advisor. You help homeowners\nunderstand their energy consumption, reduce costs, and make smarter decisions about\nappliances, solar, and utility usage. You speak in a friendly but professional tone, and\nalways prioritize actionable advice over generic tips.\n\n## Core Capabilities\n- Analyze energy usage patterns from historical data\n- Query real-time and forecasted electricity prices (time-of-use rates)\n- Retrieve weather forecasts and solar irradiance data\n- Search energy-saving best practices from a knowledge base\n- Calculate potential savings from efficiency improvements\n- Provide personalized recommendations ranked by impact and ease

## 4. Evaluate Responses

In [24]:
# TODO: Implement evaluation functions
# Create functions to evaluate:
# - Final Response
# - Tool usage

In [25]:
def evaluate_response(question, final_response, expected_response):
    """Evaluate a single response against expected criteria using four named metrics.

    Args:
        question (str): The original question asked
        final_response (str): The agent's final response text
        expected_response (str): Description of what was expected (used for ACCURACY)

    Returns:
        dict: Evaluation scores for ACCURACY, RELEVANCE, COMPLETENESS, USEFULNESS with feedback
    """
    # =============================================================================
    # ACCURACY - Uses LLM-as-judge to compare response against expected criteria
    # =============================================================================
    accuracy_prompt = f"""You are an expert evaluator judging an AI assistant's response.
    
Question asked: "{question}"

Expected response characteristics: {expected_response}

Actual response to evaluate:
{final_response}

Evaluate how ACCURATE the response is by checking if it contains the information expected above.
Rate on a scale of 1-5:
- 1: Response is incorrect or contradicts the expected information
- 2: Response addresses the topic but with significant inaccuracies or missing key information
- 3: Response is partially accurate with some gaps
- 4: Response is mostly accurate with minor omissions
- 5: Response is completely accurate and matches what was expected

Respond with ONLY a single number (1-5) and a brief reason."""

    try:
        accuracy_result = judge_llm.invoke(accuracy_prompt)
        accuracy_content = accuracy_result.content if hasattr(accuracy_result, 'content') else str(accuracy_result)
        # Extract numeric score from response
        import re
        accuracy_match = re.search(r'[1-5]', accuracy_content.strip()[:10])
        accuracy_score = int(accuracy_match.group()) if accuracy_match else 3
        accuracy_reason = accuracy_content
    except Exception as e:
        accuracy_score = 3
        accuracy_reason = f"Could not evaluate: {str(e)[:50]}"

    # =============================================================================
    # RELEVANCE - How well the response addresses the specific question
    # =============================================================================
    question_lower = question.lower()
    response_lower = final_response.lower()
    
    # Key topic keywords to check relevance
    key_topics = {
        "ev_charging": ["charge", "charging", "electric", "vehicle", "battery", "tesla", "ev"],
        "thermostat": ["thermostat", "temperature", "cooling", "heating", "ac", "hvac", "set"],
        "appliance": ["dishwasher", "washing", "machine", "appliance", "run", "schedule"],
        "solar": ["solar", "generation", "panels", "irradiance", "sun", "photovoltaic"],
        "cost": ["cost", "dollar", "price", "rate", "bill", "kwh", "save", "saving"],
        "peak": ["peak", "off-peak", "offpeak", "tier", "hours"],
        "hvac": ["hvac", "ac", "air", "conditioning", "cooling", "heating", "comfort"],
        "pool": ["pool", "pump", "swimming", "run time", "schedule"],
        "audit": ["waster", "phantom", "standby", "bulb", "led", "efficient"],
    }
    
    # Find relevant topics based on question
    relevant_topics = []
    for topic, keywords in key_topics.items():
        if any(kw in question_lower for kw in keywords):
            relevant_topics.append(topic)
    
    # Check coverage of relevant topics
    topics_covered = sum(1 for t in relevant_topics if any(kw in response_lower for kw in key_topics[t]))
    relevance_coverage = topics_covered / max(len(relevant_topics), 1)
    
    # RELEVANCE score (1-5)
    if relevance_coverage >= 0.8:
        relevance_score = 5
        relevance_feedback = "Response directly addresses all aspects of the question"
    elif relevance_coverage >= 0.6:
        relevance_score = 4
        relevance_feedback = f"Response covers most relevant topics ({topics_covered}/{len(relevant_topics)})"
    elif relevance_coverage >= 0.4:
        relevance_score = 3
        relevance_feedback = f"Response partially addresses the question ({topics_covered}/{len(relevant_topics)} topics)"
    elif relevance_coverage >= 0.2:
        relevance_score = 2
        relevance_feedback = f"Response misses many relevant topics ({topics_covered}/{len(relevant_topics)})"
    else:
        relevance_score = 1
        relevance_feedback = "Response does not address the question properly"

    # =============================================================================
    # COMPLETENESS - Does the response provide comprehensive information
    # =============================================================================
    has_numbers = any(c.isdigit() for c in final_response)
    has_recommendation = any(word in response_lower for word in ["recommend", "suggest", "should", "best", "optimal", "try", "consider", "option"])
    has_specific_data = any(word in response_lower for word in ["kwh", "watt", "hour", "dollar", "$", "percent", "%"])
    has_summary = any(word in response_lower for word in ["summary", "in summary", "to summarize", "finally", "next step", "overall"])
    
    completeness_indicators = [has_numbers, has_recommendation, has_specific_data, has_summary]
    completeness_count = sum(completeness_indicators)
    
    if completeness_count >= 4:
        completeness_score = 5
        completeness_feedback = "Response is comprehensive with numbers, recommendations, specific data, and summary"
    elif completeness_count >= 3:
        completeness_score = 4
        completeness_feedback = "Response is thorough with most key elements present"
    elif completeness_count >= 2:
        completeness_score = 3
        completeness_feedback = "Response provides some detail but lacks completeness"
    elif completeness_count >= 1:
        completeness_score = 2
        completeness_feedback = "Response is incomplete, missing key elements"
    else:
        completeness_score = 1
        completeness_feedback = "Response lacks essential information"

    # =============================================================================
    # USEFULNESS - Does the response provide actionable, helpful advice
    # =============================================================================
    has_actionable = any(word in response_lower for word in ["recommend", "suggest", "should", "try", "consider", "do", "run", "charge", "set"])
    has_savings = any(word in response_lower for word in ["save", "saving", "cost", "dollar", "$", "reduce", "lower"])
    has_tradeoff = any(word in response_lower for word in ["however", "but", "although", "tradeoff", "trade-off", "instead", "alternative"])
    has_next_step = any(word in response_lower for word in ["next", "following", "also", "additionally", "consider"])
    
    usefulness_indicators = [has_actionable, has_savings, has_tradeoff, has_next_step]
    usefulness_count = sum(usefulness_indicators)
    
    if usefulness_count >= 4:
        usefulness_score = 5
        usefulness_feedback = "Highly actionable with savings estimates, tradeoffs, and next steps"
    elif usefulness_count >= 3:
        usefulness_score = 4
        usefulness_feedback = "Useful with actionable advice and useful context"
    elif usefulness_count >= 2:
        usefulness_score = 3
        usefulness_feedback = "Somewhat useful but could be more actionable"
    elif usefulness_count >= 1:
        usefulness_score = 2
        usefulness_feedback = "Limited usefulness, needs more actionable content"
    else:
        usefulness_score = 1
        usefulness_feedback = "Not useful - lacks actionable advice"

    # =============================================================================
    # Compile all metrics into result dict
    # =============================================================================
    return {
        "accuracy": {
            "score": accuracy_score,
            "max_score": 5,
            "feedback": f"Accuracy: {accuracy_reason[:100]}..."
        },
        "relevance": {
            "score": relevance_score,
            "max_score": 5,
            "feedback": relevance_feedback
        },
        "completeness": {
            "score": completeness_score,
            "max_score": 5,
            "feedback": completeness_feedback
        },
        "usefulness": {
            "score": usefulness_score,
            "max_score": 5,
            "feedback": usefulness_feedback
        },
        "overall_score": round((accuracy_score + relevance_score + completeness_score + usefulness_score) / 4, 1),
        "max_score": 5,
        "feedback": [
            f"ACCURACY ({accuracy_score}/5): {accuracy_reason[:80]}",
            f"RELEVANCE ({relevance_score}/5): {relevance_feedback}",
            f"COMPLETENESS ({completeness_score}/5): {completeness_feedback}",
            f"USEFULNESS ({usefulness_score}/5): {usefulness_feedback}"
        ]
    }

In [26]:
def evaluate_tool_usage(messages, expected_tools):
    """Evaluate if the right tools were used.

    Args:
        messages (list): List of messages from agent response
        expected_tools (list): List of tool names that were expected to be called

    Returns:
        dict: Evaluation score and details
    """
    # Extract actual tool calls from messages
    actual_tools = []
    for msg in messages:
        msg_dict = msg.model_dump() if hasattr(msg, 'model_dump') else msg
        if msg_dict.get("tool_call_id"):
            # This is a tool result - get the tool name
            tool_name = msg.name if hasattr(msg, 'name') else msg_dict.get("name", "unknown")
            actual_tools.append(tool_name)

    score = 0
    max_score = len(expected_tools) if len(expected_tools) > 0 else 1
    feedback = []

    # Check if expected tools were called
    if expected_tools:
        matched = [t for t in expected_tools if t in actual_tools]
        missing = [t for t in expected_tools if t not in actual_tools]
        extra = [t for t in actual_tools if t not in expected_tools]

        score = len(matched)
        max_score = len(expected_tools)

        if matched:
            feedback.append(f"✓ Correct tools used: {matched}")
        if missing:
            feedback.append(f"✗ Missing tools: {missing}")
        if extra:
            feedback.append(f"ℹ Extra tools used: {extra}")

        percentage = round(score / max_score * 100, 1) if max_score > 0 else 0
    else:
        percentage = 100 if not actual_tools else 50
        feedback.append("No specific tools expected")

    return {
        "score": score,
        "max_score": max_score,
        "percentage": percentage,
        "expected_tools": expected_tools,
        "actual_tools": actual_tools,
        "feedback": feedback
    }

In [27]:
def generate_evaluation_report(test_results):
    """Generate a comprehensive evaluation report.

    Args:
        test_results (list): List of test results from agent runs

    Returns:
        dict: Comprehensive evaluation report with metrics and recommendations
    """
    report = {
        "total_tests": len(test_results),
        "metrics": {
            "accuracy": {"scores": [], "average": 0},
            "relevance": {"scores": [], "average": 0},
            "completeness": {"scores": [], "average": 0},
            "usefulness": {"scores": [], "average": 0},
            "overall": {"scores": [], "average": 0}
        },
        "tool_usage_evaluation": {
            "total_score": 0,
            "total_max": 0,
            "average_percentage": 0,
            "details": []
        },
        "response_details": [],
        "strengths": [],
        "weaknesses": [],
        "recommendations": []
    }

    tool_scores = []

    for result in test_results:
        question = result.get("question", "")
        expected_response = result.get("expected_response", "")
        expected_tools = result.get("expected_tools", [])

        # Get response messages
        if "error" in str(result.get("response", "")).lower():
            # Error case - skip evaluation
            continue

        response_obj = result.get("response", {})
        messages = response_obj.get("messages", [])

        # Get final response (last AI message)
        final_response = ""
        for msg in reversed(messages):
            if hasattr(msg, "type") and msg.type == "ai":
                final_response = msg.content
                break
            elif hasattr(msg, "content") and "content" in msg.additional_kwargs:
                # Check if it's an AI message
                if not hasattr(msg, "tool_call_id"):
                    final_response = msg.content
                    break

        # Evaluate response quality with four named metrics
        resp_eval = evaluate_response(question, final_response, expected_response)
        
        # Store detailed response evaluation
        report["response_details"].append({
            "test_id": result["test_id"],
            **resp_eval
        })
        
        # Extract individual metric scores
        report["metrics"]["accuracy"]["scores"].append(resp_eval["accuracy"]["score"])
        report["metrics"]["relevance"]["scores"].append(resp_eval["relevance"]["score"])
        report["metrics"]["completeness"]["scores"].append(resp_eval["completeness"]["score"])
        report["metrics"]["usefulness"]["scores"].append(resp_eval["usefulness"]["score"])
        report["metrics"]["overall"]["scores"].append(resp_eval["overall_score"])

        # Evaluate tool usage
        tool_eval = evaluate_tool_usage(messages, expected_tools)
        report["tool_usage_evaluation"]["details"].append({
            "test_id": result["test_id"],
            **tool_eval
        })
        tool_scores.append(tool_eval["percentage"])

    # Calculate averages for each metric
    for metric in ["accuracy", "relevance", "completeness", "usefulness", "overall"]:
        scores = report["metrics"][metric]["scores"]
        if scores:
            report["metrics"][metric]["average"] = round(sum(scores) / len(scores), 1)
        else:
            report["metrics"][metric]["average"] = 0

    # Tool usage average
    if tool_scores:
        report["tool_usage_evaluation"]["average_percentage"] = round(sum(tool_scores) / len(tool_scores), 1)

    # Identify strengths based on metrics
    if report["metrics"]["accuracy"]["average"] >= 3.5:
        report["strengths"].append("Responses are accurate and match expected criteria")
    if report["metrics"]["relevance"]["average"] >= 3.5:
        report["strengths"].append("Responses directly address user questions")
    if report["metrics"]["completeness"]["average"] >= 3.5:
        report["strengths"].append("Responses provide comprehensive information")
    if report["metrics"]["usefulness"]["average"] >= 3.5:
        report["strengths"].append("Responses contain actionable advice")
    if report["tool_usage_evaluation"]["average_percentage"] >= 70:
        report["strengths"].append("Appropriate tool usage")

    # Identify weaknesses
    if report["metrics"]["accuracy"]["average"] < 3:
        report["weaknesses"].append("Accuracy needs improvement - check response alignment with expected criteria")
    if report["metrics"]["relevance"]["average"] < 3:
        report["weaknesses"].append("Relevance needs improvement - ensure responses match questions")
    if report["metrics"]["completeness"]["average"] < 3:
        report["weaknesses"].append("Completeness needs improvement - add more details and data")
    if report["metrics"]["usefulness"]["average"] < 3:
        report["weaknesses"].append("Usefulness needs improvement - include more actionable recommendations")
    if report["tool_usage_evaluation"]["average_percentage"] < 70:
        report["weaknesses"].append("Tool selection needs improvement")

    # Generate recommendations
    if report["metrics"]["overall"]["average"] < 3:
        report["recommendations"].append("Focus on improving overall response quality")
    if report["metrics"]["accuracy"]["average"] < 3:
        report["recommendations"].append("Review expected response criteria and ensure alignment")
    if report["tool_usage_evaluation"]["average_percentage"] < 50:
        report["recommendations"].append("Review tool selection logic in agent instructions")
    
    report["recommendations"].append("Consider adding more test cases for edge cases")
    report["recommendations"].append("Implement human feedback collection for real-world validation")

    return report


def display_evaluation_report(report):
    """Display the evaluation report in a formatted way.
    
    This function separates report generation from display, as required by the rubric.
    
    Args:
        report (dict): The evaluation report generated by generate_evaluation_report()
    """
    print("=" * 70)
    print("EVALUATION REPORT")
    print("=" * 70)
    print(f"\nTotal Tests: {report['total_tests']}")
    
    # Display metrics
    print("\n" + "=" * 70)
    print("RESPONSE QUALITY METRICS")
    print("=" * 70)
    metrics = report["metrics"]
    print(f"  ACCURACY:    {metrics['accuracy']['average']}/5.0")
    print(f"  RELEVANCE:   {metrics['relevance']['average']}/5.0")
    print(f"  COMPLETENESS:{metrics['completeness']['average']}/5.0")
    print(f"  USEFULNESS:  {metrics['usefulness']['average']}/5.0")
    print(f"  OVERALL:     {metrics['overall']['average']}/5.0")
    
    print(f"\nTool Usage: {report['tool_usage_evaluation']['average_percentage']}%")
    
    # Display strengths
    print("\n" + "-" * 70)
    print("STRENGTHS")
    print("-" * 70)
    if report["strengths"]:
        for s in report["strengths"]:
            print(f"  + {s}")
    else:
        print("  No strengths identified")
    
    # Display weaknesses
    print("\n" + "-" * 70)
    print("WEAKNESSES")
    print("-" * 70)
    if report["weaknesses"]:
        for w in report["weaknesses"]:
            print(f"  - {w}")
    else:
        print("  No weaknesses identified")
    
    # Display recommendations
    print("\n" + "-" * 70)
    print("RECOMMENDATIONS")
    print("-" * 70)
    for r in report["recommendations"]:
        print(f"  > {r}")
    
    # Display per-test details
    print("\n" + "=" * 70)
    print("PER-TEST DETAILS")
    print("=" * 70)
    for detail in report["response_details"]:
        print(f"\nTest: {detail['test_id']}")
        print(f"  Overall: {detail['overall_score']}/5.0")
        for fb in detail.get("feedback", []):
            print(f"  {fb[:65]}..." if len(fb) > 65 else f"  {fb}")

In [28]:
# Generate evaluation report using the dedicated function
evaluation_report = generate_evaluation_report(test_results)

# Display the report using the dedicated display function
display_evaluation_report(evaluation_report)

EVALUATION REPORT

Total Tests: 10

RESPONSE QUALITY METRICS
  ACCURACY:    4.7/5.0
  RELEVANCE:   5.0/5.0
  COMPLETENESS:4.6/5.0
  USEFULNESS:  4.0/5.0
  OVERALL:     4.6/5.0

Tool Usage: 20.0%

----------------------------------------------------------------------
STRENGTHS
----------------------------------------------------------------------
  + Responses are accurate and match expected criteria
  + Responses directly address user questions
  + Responses provide comprehensive information
  + Responses contain actionable advice

----------------------------------------------------------------------
WEAKNESSES
----------------------------------------------------------------------
  - Tool selection needs improvement

----------------------------------------------------------------------
RECOMMENDATIONS
----------------------------------------------------------------------
  > Review tool selection logic in agent instructions
  > Consider adding more test cases for edge cases
  > Impl